![Banner](./img/AI_Special_Program_Banner.jpg)

## Introduction to Transformer Models - Exercise
---

The exercise you are going to do this time is slightly different from the previous ones.
Your task will be to find out more about Transformer Model on your own. To do this, you will read the paper that is the cornerstone of many modern Transformer applications. The paper, titled **Attention Is All You Need**, was published by Google scientists in 2017 and can be downloaded [here](https://arxiv.org/abs/1706.03762).

If some of the terms in the paper sound unfamiliar to you, don't hesitate to consult other sources on the internet. It is also recommended to watch the following video, which explains the components of a Transformer model, step by step, in an excellent way:
* [Illustrated Guide to Transformers](https://www.youtube.com/watch?v=4Bdc55j80l8)

After you have done your research, try to answer the following questions:

Answer:

## Question 1: What does self-attention do in the context of Transformers?

Answer:

**In one sentence:** self-attention re-writes every token of a sequence as a weighted average of *all* tokens of that same sequence, so that a representation which started out context-free becomes context-aware.

**The mechanism.** Each token representation is projected by three learned linear layers into a **query**, a **key** and a **value**. Every query is compared against every key by a dot-product, the scores are normalised by a softmax, and the result for that query is the weighted average of the values in exactly those proportions:

$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$

The division by $\sqrt{d_k}$ is the "**scaled**" part. Dot-products of high-dimensional vectors grow with the number of dimensions, which would push the softmax into a saturated region where gradients nearly vanish; the scaling keeps it in a trainable regime.

The word "**self**" simply means the query sequence and the key-value sequence are *the same* sequence. In the decoder's cross-attention sublayer the queries come from the German sequence while keys and values come from the encoded English one — same mechanism, but no longer *self*-attention.

**What it accomplishes.** A token embedding taken from a look-up table represents that token alone, with no situational awareness: `bank` has one vector whether the sentence is about a river or about money. After a self-attention layer the representation at that position is a blend of the whole sentence, weighted by relevance — so `bank` next to `river` and `bank` next to `deposit` become different vectors. Stacking 6 such layers builds context upon context, which is where the transformer's representational power comes from.

**Why this beats recurrence** — two structural properties, both highlighted in Section 4 of the paper:

* **Parallelism.** The result for one query does not depend on the result for any other query, so all positions are computed simultaneously. An RNN cannot reach position $t$ without having processed $t-1$ first. This is what made training on very large corpora practical.
* **Constant path length.** In self-attention the number of operations separating any two positions is $O(1)$ — every token has *direct* access to every other. In an RNN that distance is $O(n)$, and information must survive the whole daisy-chain of intervening steps; the LSTM's cell state mitigates this but does not remove it. Long-range dependencies therefore become much easier to learn.

**Multi-head.** A single attention operation yields one interpretation of the relationships in a sentence, but there are many valid ones simultaneously (subject–verb agreement, adjective–noun modification, coreference, ...). So the transformer runs $h=8$ attention operations in parallel, each in its own learned subspace of $d_\text{queries}=d_\text{keys}=d_\text{values}=64$ dimensions, then concatenates the results and projects them back to $d_\text{model}=512$. Because $8\times 64=512$, the total cost is roughly that of one full-width head — the heads are free in FLOPs and trivially parallelisable.

**One important caveat.** Self-attention is *permutation-equivariant*: it treats its input as a bag of tokens. On its own it cannot tell "dog bites man" from "man bites dog", which is precisely why positional encodings are needed (Question 2). And in the decoder the attention must additionally be **masked**, so that a position may attend only to positions at or before itself — otherwise the model would be allowed to look at the very tokens it is supposed to predict.

## Question 2: What are positional encodings and why are they needed?

Answer:

**Why they are needed.** A transformer has no recurrence and no convolution, so nothing in the architecture knows *where* a token sits. Attention computes a weighted average over a set, and averaging is order-blind: if you shuffle the input tokens, the outputs come back shuffled in exactly the same way but otherwise unchanged. Formally, self-attention is **permutation-equivariant**. Without extra information the model would see "the dog bit the man" and "the man bit the dog" as the same bag of tokens — a fatal problem for any language task, where word order carries much of the meaning.

An RNN never has this problem, because position is *implicit* in the order in which it processes the sequence. That convenience is exactly what the transformer gave up in exchange for parallelism, so **order has to be injected back in explicitly**.

**What they are.** A positional encoding is a vector of the same size $d_\text{model}=512$ as the token embedding, one per position in the sequence, which is **added** to the token embedding before the first encoder/decoder layer:

$$x_{\text{pos}} = \text{TokenEmbedding}(\text{token}) + \text{PositionalEncoding}(\text{pos})$$

Adding rather than concatenating keeps the dimensionality at $d_\text{model}$ throughout the network and costs no extra parameters; because the token embeddings are learned, the model can organise itself so that the two kinds of information do not destructively interfere.

**The paper's choice: fixed sinusoids.** Rather than learning these vectors, the authors define them with sine and cosine functions of geometrically increasing wavelengths:

$$PE_{(pos,\,2i)}=\sin\!\left(\frac{pos}{10000^{2i/d_\text{model}}}\right),\qquad
PE_{(pos,\,2i+1)}=\cos\!\left(\frac{pos}{10000^{2i/d_\text{model}}}\right)$$

Each dimension corresponds to a sinusoid whose wavelength runs from $2\pi$ up to $10000\cdot 2\pi$, so every position receives a unique pattern — rather like a binary encoding of the position, but continuous. This is exactly what the `PositionalEncoding` class in the [machine-translation notebook](3.3b_Machine_Translation_HandsOn.ipynb) implements: `div_term` builds the geometric progression of frequencies, `sin` fills the even dimensions and `cos` the odd ones, and the result is stored with `register_buffer` — it is part of the model's state but has **no trainable parameters**.

**Why sinusoids in particular?** The paper gives two reasons:

* **Relative positions become easy to express.** For any fixed offset $k$, $PE_{pos+k}$ is a *linear* function of $PE_{pos}$ — it is just a rotation, by the angle-addition formulas for sine and cosine. The model can therefore learn to attend by relative distance ("three tokens back") with a simple linear transformation, which is often what grammar actually depends on.
* **Extrapolation to unseen lengths.** The formula is defined for any position, so a model trained on short sentences can at least be *evaluated* on longer ones. A learned look-up table has no entry beyond its maximum length.

The authors also tried **learned** positional embeddings and reported nearly identical results (Table 3, row E); they kept the sinusoidal version for the extrapolation property. Interestingly, most models that followed went the other way: BERT and GPT-2 use learned absolute positions, and the `TransformerModel` class in notebook 3.3b likewise uses a learned encoding initialised to zeros.

**Where the field went next.** Absolute encodings have a weakness — they tell the model where a token is, but attention usually cares about how far apart two tokens are. Later work therefore moved to genuinely relative schemes: **relative position biases** added to the attention scores (T5), **RoPE** (rotary embeddings, used by Llama and most current open models), which rotates queries and keys so that their dot-product depends only on the relative offset, and **ALiBi**, which simply penalises attention by distance. All of them address the same fundamental need identified in 2017: attention alone cannot see order.

## Question 3: What are some of the limitations/challenges associated with Transformer Models?

Answer:

**1. Quadratic cost in sequence length — the central limitation.** Every token attends to every other, so the attention matrix has $n^2$ entries and self-attention costs $O(n^2\cdot d)$ time and memory. Table 1 of the paper presents this as an *advantage*: as long as $n < d$, self-attention is cheaper per layer than recurrence ($O(n\cdot d^2)$), which holds comfortably for sentence-length translation. But the picture inverts for long inputs — books, high-resolution images, genomes — where doubling the context quadruples the cost. This single property has driven a whole research literature: sparse and windowed attention (Longformer, BigBird), low-rank and kernel approximations (Linformer, Performer), IO-aware exact implementations (FlashAttention, which does not reduce the FLOPs but avoids materialising the $n\times n$ matrix in memory), and more recently state-space models such as Mamba that abandon attention for long contexts altogether.

**2. Weak inductive bias, hence data and compute hunger.** A CNN builds in locality and translation equivariance; an RNN builds in sequential order. A transformer assumes essentially nothing — which is exactly why it scales so well, but it means that everything, including the notion that neighbouring words are related, must be *learned from data*. On small datasets this is a real disadvantage: the Vision Transformer paper is the clearest demonstration, with ViT underperforming a comparable ResNet when trained on ImageNet alone and overtaking it only after pretraining on the 300-million-image JFT dataset. The practical consequence is the pretrain-then-finetune paradigm that now dominates the field.

**3. Cost, energy and accessibility.** The paper's own models were modest — 65M parameters for the base, 213M for the big one, 3.5 days on 8 GPUs — but the scaling that transformers enabled has made frontier training runs cost millions of dollars and consume correspondingly large amounts of energy. This concentrates capability in a few well-funded organisations and raises genuine environmental questions.

**4. Inference is still sequential, and the KV cache grows.** Training is fully parallel thanks to teacher forcing, but *generation* remains autoregressive: token $t+1$ cannot be produced before token $t$. So the headline parallelism advantage applies to training, not to decoding. Worse, efficient decoding requires caching the keys and values of every previous token, and that cache grows linearly with context length and quickly dominates GPU memory in long conversations.

**5. A bounded context window.** In practice a model is trained at some maximum length and degrades outside it. Fixed sinusoids extrapolate in principle, but quality still falls off, and learned positional tables have no entries at all beyond their table size. Extending context reliably remains an active problem.

**6. Interpretability is not what it appears to be.** It is tempting to read attention weights as an explanation of the model's decision, and the paper itself shows such visualisations. But with 8 heads across 6 layers, plus residual connections and feed-forward mixing, the information routed to a position is not well summarised by any single attention map — a body of later work (*"Attention is not Explanation"*) shows that attention weights can be substantially altered without changing the prediction. Attention maps are a useful diagnostic, not a faithful explanation.

**7. Failure modes inherited by the applications.** Because a language model is trained only to predict a plausible next token, nothing in the objective enforces truth: the same sampling that makes generation creative also produces confident, fluent falsehoods (**hallucination**) — the effect already visible in miniature in the [tiny language model notebook](3.2_RNN_3_TLM_HandsOn.ipynb). Models likewise absorb the **biases** of their training corpora, and there is no built-in mechanism for keeping knowledge current.

**8. Training can be finicky.** The original recipe depends on a learning-rate schedule with a warmup phase (4000 steps in the paper); training a post-normalisation transformer without it is unstable, which is why most modern implementations moved the layer normalisation to the front of each sublayer — the arrangement the [theory notebook](3.3a_Transformer_Theory.ipynb) describes.

**In short:** the transformer trades built-in structure for generality and parallelism. That trade is what made large-scale pretraining work, and nearly every limitation above is the bill for it — either the quadratic price of letting every token see every other, or the price of assuming nothing and having to learn it all from data.